# 04. 추론 Loop: BERT PM + LLM 분류 + Null Intent 탐지

**목적**: 초기 IntentSet(907 클러스터)과 Discriminative Description을 활용하여 의도 분류 시스템을 구축하고 성능을 측정한다.

**방법론**: FCSLM + IntentGPT 아이디어를 결합한 1회 Loop.
1. BERT PM(Prediction Model)을 train split으로 학습하여 907 클래스 분류기 구축
2. Val split에서 BERT PM의 confidence 기반으로 고신뢰/저신뢰 분리
3. 고신뢰 쿼리 → BERT가 직접 분류 확정
4. 저신뢰 쿼리 → Gemini + Discriminative Description으로 LLM 분류
5. BERT/LLM 예측 불일치 시 → Comparative Reasoning으로 최종 결정
6. Null intent 탐지 → 기존 의도에 해당하지 않는 쿼리를 새 의도 후보로 분리

**모델**: `klue/roberta-base` (BERT PM), `gemini-2.5-flash-lite` (LLM)

**산출물**:
- `checkpoints/loop/bert_pm/` — BERT PM 모델 체크포인트
- `data/processed/loop_results.parquet` — 전체 분류 결과
- `data/processed/loop_evaluation.json` — 성능 메트릭

**예상 비용**: BERT 학습 ₩0 (GPU) + LLM 분류 ~₩170 / 잔여 ~₩43,600

**예상 소요**: BERT 학습 ~25분 + 추론 ~15분 + LLM 분류 ~10분

## 0. 환경 설정
GPU 런타임 필요 (BERT PM 학습 및 추론).

In [1]:
# === 환경 설정 ===
from google.colab import drive
drive.mount('/content/drive')

import os
os.chdir('/content/drive/MyDrive/5stone_experiment/1_clustering_test')
print(f'작업 디렉토리: {os.getcwd()}')

!pip install -q google-genai transformers datasets accelerate

import json
import time
import unicodedata
import numpy as np
import pandas as pd
from pathlib import Path
from collections import Counter
from tqdm.notebook import tqdm
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"})')

Mounted at /content/drive
작업 디렉토리: /content/drive/MyDrive/5stone_experiment/1_clustering_test
Device: cuda (Tesla T4)


## 0-1. 경로 및 하이퍼파라미터 설정

In [2]:
# === 경로 설정 ===
BASE_DIR = Path('/content/drive/MyDrive/5stone_experiment/1_clustering_test')

PATH_CONFIG = {
    'processed': BASE_DIR / 'data' / 'processed',
    'checkpoints': BASE_DIR / 'checkpoints',
}

# 입력
PATH_CONFIG['initial_intentset'] = PATH_CONFIG['processed'] / 'initial_intentset.parquet'
PATH_CONFIG['intent_descriptions'] = PATH_CONFIG['processed'] / 'intent_descriptions.parquet'

# 산출물
PATH_CONFIG['bert_pm_dir'] = PATH_CONFIG['checkpoints'] / 'loop' / 'bert_pm'
PATH_CONFIG['loop_results'] = PATH_CONFIG['processed'] / 'loop_results.parquet'
PATH_CONFIG['loop_evaluation'] = PATH_CONFIG['processed'] / 'loop_evaluation.json'
PATH_CONFIG['loop_ckpt_dir'] = PATH_CONFIG['checkpoints'] / 'loop'

PATH_CONFIG['bert_pm_dir'].mkdir(parents=True, exist_ok=True)
PATH_CONFIG['loop_ckpt_dir'].mkdir(parents=True, exist_ok=True)

# === 하이퍼파라미터 ===
CONFIG = {
    # --- BERT PM ---
    'bert_model_name': 'klue/roberta-base',
    'bert_max_length': 128,
    'bert_epochs': 5,
    'bert_batch_size': 32,
    'bert_lr': 2e-5,
    'bert_weight_decay': 0.01,
    'bert_warmup_ratio': 0.1,

    # --- Confidence 분할 ---
    'confidence_threshold': 0.7,   # 이 이상이면 고신뢰
    'top_k_candidates': 5,         # LLM에 전달할 후보 의도 수

    # --- Gemini API ---
    'gemini_model': 'gemini-2.5-flash-lite',
    'gemini_temperature': 0.1,
    'gemini_max_output_tokens': 256,
    'gemini_rate_limit_delay': 0.25,
    'gemini_retry_max': 3,
    'gemini_retry_delay': 5,

    # --- Null Intent ---
    'null_intent_enabled': True,
    'null_confidence_max': 0.3,    # 모든 후보의 confidence가 이 미만이면 null 의심
}

for key in ['initial_intentset', 'intent_descriptions']:
    marker = '✓' if PATH_CONFIG[key].exists() else '✗'
    print(f'  {marker} {key}')
print(f'\nBERT: {CONFIG["bert_model_name"]}, epochs={CONFIG["bert_epochs"]}, batch={CONFIG["bert_batch_size"]}')
print(f'Confidence threshold: {CONFIG["confidence_threshold"]}')

  ✓ initial_intentset
  ✓ intent_descriptions

BERT: klue/roberta-base, epochs=5, batch=32
Confidence threshold: 0.7


## 0-2. Gemini API 초기화

In [3]:
# === Gemini API 초기화 ===
from google.colab import userdata
from google import genai
from google.genai import types

client = genai.Client(api_key=userdata.get('GOOGLE_API_KEY'))

def nfc(text: str) -> str:
    return unicodedata.normalize('NFC', text)

def call_gemini(prompt: str, system_instruction: str) -> str:
    for attempt in range(CONFIG['gemini_retry_max']):
        try:
            response = client.models.generate_content(
                model=CONFIG['gemini_model'],
                contents=prompt,
                config=types.GenerateContentConfig(
                    system_instruction=system_instruction,
                    temperature=CONFIG['gemini_temperature'],
                    max_output_tokens=CONFIG['gemini_max_output_tokens'],
                    response_mime_type='application/json',
                ),
            )
            time.sleep(CONFIG['gemini_rate_limit_delay'])
            return response.text.strip()
        except Exception as e:
            error_str = str(e)
            if '429' in error_str or 'RESOURCE_EXHAUSTED' in error_str:
                wait = CONFIG['gemini_retry_delay'] * (attempt + 1) * 2
                time.sleep(wait)
            else:
                wait = CONFIG['gemini_retry_delay'] * (attempt + 1)
                time.sleep(wait)
    return ''

test_resp = client.models.generate_content(
    model=CONFIG['gemini_model'],
    contents='테스트. "ok"만 답하세요.',
    config=types.GenerateContentConfig(temperature=0.0, max_output_tokens=10),
)
print(f'Gemini OK: {test_resp.text}')

Gemini OK: ok


## 1. 데이터 로드 및 Train/Test 분할
IntentSet(907 클러스터)과 Description을 로드하고, 원본 split 기반으로 train/test를 나눈다.

- **Train**: 원본 split='train'인 의도 레코드 → BERT PM 학습
- **Test**: 원본 split='val'인 의도 레코드 → Loop 추론 대상

In [4]:
# === 데이터 로드 및 분할 ===

def load_loop_data() -> tuple:
    """
    IntentSet과 Description을 로드하고, train/test로 분할한다.
    할당된(cluster_id >= 0) 레코드만 사용.
    반환: (train_df, test_df, desc_df, label2id, id2label)
    """
    # IntentSet
    intent_df = pd.read_parquet(PATH_CONFIG['initial_intentset'])
    intent_df['intent_summary'] = intent_df['intent_summary'].apply(nfc)
    intent_df['intent_label'] = intent_df['intent_label'].apply(nfc)

    # 할당된 레코드만
    assigned = intent_df[intent_df['cluster_id'] >= 0].copy()
    print(f'전체: {len(intent_df):,}, 할당됨: {len(assigned):,}')

    # 입력 텍스트 구성: intent_summary + relevant_utterances
    def build_input_text(row):
        text = row['intent_summary']
        try:
            utts = json.loads(row['relevant_utterances'])
            if utts:
                text += ' [SEP] ' + ' '.join(utts[:3])  # 최대 3개 발화
        except (json.JSONDecodeError, TypeError):
            pass
        return nfc(text)

    assigned['input_text'] = assigned.apply(build_input_text, axis=1)

    # 라벨 매핑
    unique_labels = sorted(assigned['intent_label'].unique())
    label2id = {label: idx for idx, label in enumerate(unique_labels)}
    id2label = {idx: label for label, idx in label2id.items()}
    assigned['label_id'] = assigned['intent_label'].map(label2id)

    print(f'고유 라벨: {len(unique_labels)}')

    # Train/Test 분할 (원본 split 기반)
    train_df = assigned[assigned['split'] == 'train'].copy()
    test_df = assigned[assigned['split'] == 'val'].copy()

    print(f'Train: {len(train_df):,}, Test: {len(test_df):,}')

    # Test에만 존재하는 라벨 확인 (train에 없으면 분류 불가)
    train_labels = set(train_df['intent_label'].unique())
    test_labels = set(test_df['intent_label'].unique())
    test_only = test_labels - train_labels
    if test_only:
        print(f'  ⚠ Test에만 존재하는 라벨: {len(test_only)}개 (null intent 후보)')

    # Description
    desc_df = pd.read_parquet(PATH_CONFIG['intent_descriptions'])
    desc_df['intent'] = desc_df['intent'].apply(nfc)
    desc_df['definition'] = desc_df['definition'].apply(nfc)
    print(f'Description: {len(desc_df)}개 (fallback: {desc_df["is_fallback"].sum()})')

    return train_df, test_df, desc_df, label2id, id2label


train_df, test_df, desc_df, label2id, id2label = load_loop_data()

# 입력 텍스트 길이 확인
print(f'\n입력 텍스트 길이 (글자 수):')
print(f'  Train: mean={train_df["input_text"].str.len().mean():.0f}, max={train_df["input_text"].str.len().max()}')
print(f'  Test:  mean={test_df["input_text"].str.len().mean():.0f}, max={test_df["input_text"].str.len().max()}')

전체: 34,025, 할당됨: 33,341
고유 라벨: 907
Train: 29,604, Test: 3,737
  ⚠ Test에만 존재하는 라벨: 4개 (null intent 후보)
Description: 907개 (fallback: 79)

입력 텍스트 길이 (글자 수):
  Train: mean=100, max=490
  Test:  mean=100, max=389


## 2. BERT PM 학습
`klue/roberta-base`를 907 클래스 분류기로 fine-tuning한다.

**체크포인트**: `checkpoints/loop/bert_pm/` — 학습된 모델 가중치

In [5]:
# === BERT PM 데이터셋 ===

class IntentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length):
        self.encodings = tokenizer(
            texts, truncation=True, padding='max_length',
            max_length=max_length, return_tensors='pt',
        )
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item['labels'] = self.labels[idx]
        return item


print('IntentDataset 정의 완료')

IntentDataset 정의 완료


In [6]:
# === BERT PM 학습 ===

def train_bert_pm(train_df: pd.DataFrame, label2id: dict) -> tuple:
    """
    klue/roberta-base를 907 클래스 분류기로 fine-tuning한다.
    체크포인트가 존재하면 로드.
    반환: (model, tokenizer)
    """
    model_dir = PATH_CONFIG['bert_pm_dir']
    model_path = model_dir / 'pytorch_model.bin'

    tokenizer = AutoTokenizer.from_pretrained(CONFIG['bert_model_name'])

    # 체크포인트 확인
    if model_path.exists() or (model_dir / 'model.safetensors').exists():
        print(f'BERT PM 체크포인트 로드: {model_dir}')
        model = AutoModelForSequenceClassification.from_pretrained(
            str(model_dir), num_labels=len(label2id),
        )
        model.to(device)
        model.eval()
        return model, tokenizer

    print(f'BERT PM 학습 시작 ({CONFIG["bert_model_name"]}, {len(label2id)} 클래스)')

    # 모델 초기화
    model = AutoModelForSequenceClassification.from_pretrained(
        CONFIG['bert_model_name'], num_labels=len(label2id),
    )

    # 데이터셋
    train_texts = train_df['input_text'].tolist()
    train_labels = train_df['label_id'].tolist()
    train_dataset = IntentDataset(train_texts, train_labels, tokenizer, CONFIG['bert_max_length'])

    # Train에서 10% val split (학습 모니터링용)
    val_size = int(len(train_dataset) * 0.1)
    train_size = len(train_dataset) - val_size
    train_subset, val_subset = torch.utils.data.random_split(
        train_dataset, [train_size, val_size],
        generator=torch.Generator().manual_seed(42),
    )

    # 학습 설정
    training_args = TrainingArguments(
        output_dir=str(model_dir / 'trainer_output'),
        num_train_epochs=CONFIG['bert_epochs'],
        per_device_train_batch_size=CONFIG['bert_batch_size'],
        per_device_eval_batch_size=CONFIG['bert_batch_size'] * 2,
        learning_rate=CONFIG['bert_lr'],
        weight_decay=CONFIG['bert_weight_decay'],
        warmup_ratio=CONFIG['bert_warmup_ratio'],
        eval_strategy='epoch',
        save_strategy='epoch',
        load_best_model_at_end=True,
        metric_for_best_model='eval_loss',
        logging_steps=50,
        fp16=torch.cuda.is_available(),
        report_to='none',
        save_total_limit=2,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_subset,
        eval_dataset=val_subset,
    )

    trainer.train()

    # 최종 모델 저장
    model.save_pretrained(str(model_dir))
    tokenizer.save_pretrained(str(model_dir))
    print(f'BERT PM 저장: {model_dir}')

    model.eval()
    return model, tokenizer


bert_model, tokenizer = train_bert_pm(train_df, label2id)

config.json:   0%|          | 0.00/546 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/375 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

BERT PM 학습 시작 (klue/roberta-base, 907 클래스)


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

BERT PM 저장: /content/drive/MyDrive/5stone_experiment/1_clustering_test/checkpoints/loop/bert_pm


## 3. BERT PM 추론 + Confidence 계산
Test split에서 BERT PM을 돌려 Top-K 후보 + confidence를 산출한다.
다중 예측 헤드 투표 대신, softmax 확률의 max 값을 confidence로 사용한다.

In [7]:
# === BERT PM 추론 ===

def bert_inference(model, tokenizer, test_df: pd.DataFrame, id2label: dict) -> pd.DataFrame:
    """
    BERT PM으로 test_df를 추론하여 Top-K 후보 + confidence를 산출한다.
    반환: test_df에 'bert_top1', 'bert_confidence', 'bert_top_k' 컬럼 추가
    """
    ckpt_path = PATH_CONFIG['loop_ckpt_dir'] / 'bert_predictions.parquet'
    if ckpt_path.exists():
        print(f'BERT 예측 체크포인트 로드: {ckpt_path}')
        return pd.read_parquet(ckpt_path)

    model.eval()
    top_k = CONFIG['top_k_candidates']

    all_results = []
    texts = test_df['input_text'].tolist()

    # 배치 추론
    batch_size = CONFIG['bert_batch_size'] * 2
    for start in tqdm(range(0, len(texts), batch_size), desc='BERT 추론'):
        batch_texts = texts[start:start + batch_size]
        inputs = tokenizer(
            batch_texts, truncation=True, padding='max_length',
            max_length=CONFIG['bert_max_length'], return_tensors='pt',
        ).to(device)

        with torch.no_grad():
            logits = model(**inputs).logits
            probs = F.softmax(logits, dim=-1)

        # Top-K
        topk_probs, topk_ids = torch.topk(probs, k=min(top_k, probs.shape[1]), dim=-1)

        for i in range(len(batch_texts)):
            top1_id = topk_ids[i][0].item()
            top1_conf = topk_probs[i][0].item()

            top_k_list = []
            for j in range(min(top_k, topk_ids.shape[1])):
                cid = topk_ids[i][j].item()
                prob = topk_probs[i][j].item()
                top_k_list.append({
                    'label_id': cid,
                    'label': id2label.get(cid, 'UNKNOWN'),
                    'prob': round(prob, 4),
                })

            all_results.append({
                'bert_top1_id': top1_id,
                'bert_top1_label': id2label.get(top1_id, 'UNKNOWN'),
                'bert_confidence': round(top1_conf, 4),
                'bert_top_k': json.dumps(top_k_list, ensure_ascii=False),
            })

    result_df = test_df.reset_index(drop=True).copy()
    pred_df = pd.DataFrame(all_results)
    result_df = pd.concat([result_df, pred_df], axis=1)

    result_df.to_parquet(ckpt_path, index=False)
    print(f'BERT 예측 저장: {ckpt_path}')

    return result_df


test_with_bert = bert_inference(bert_model, tokenizer, test_df, id2label)

# Confidence 분포
print(f'\n=== Confidence 분포 ===')
confs = test_with_bert['bert_confidence']
print(f'  평균: {confs.mean():.3f}, 중앙값: {confs.median():.3f}')
for t in [0.3, 0.5, 0.7, 0.8, 0.9]:
    n = (confs >= t).sum()
    print(f'  ≥ {t}: {n:,} ({n/len(confs)*100:.1f}%)')

# BERT Top-1 정확도 (ground truth = cluster_id 기반 intent_label)
correct = (test_with_bert['bert_top1_label'] == test_with_bert['intent_label']).sum()
total = len(test_with_bert)
print(f'\nBERT Top-1 정확도: {correct}/{total} = {correct/total*100:.1f}%')

BERT 추론:   0%|          | 0/59 [00:00<?, ?it/s]

BERT 예측 저장: /content/drive/MyDrive/5stone_experiment/1_clustering_test/checkpoints/loop/bert_predictions.parquet

=== Confidence 분포 ===
  평균: 0.211, 중앙값: 0.131
  ≥ 0.3: 924 (24.7%)
  ≥ 0.5: 439 (11.7%)
  ≥ 0.7: 168 (4.5%)
  ≥ 0.8: 55 (1.5%)
  ≥ 0.9: 13 (0.3%)

BERT Top-1 정확도: 1806/3737 = 48.3%


## 4. Confidence 기반 고신뢰/저신뢰 분리
BERT confidence가 threshold 이상이면 고신뢰(BERT 직접 확정), 미만이면 저신뢰(LLM 분류 대상).

In [8]:
# === 고신뢰/저신뢰 분리 ===

def split_by_confidence(df: pd.DataFrame) -> tuple:
    threshold = CONFIG['confidence_threshold']
    high_conf = df[df['bert_confidence'] >= threshold].copy()
    low_conf = df[df['bert_confidence'] < threshold].copy()

    print(f'=== Confidence 분할 (threshold={threshold}) ===')
    print(f'  고신뢰: {len(high_conf):,} ({len(high_conf)/len(df)*100:.1f}%)')
    print(f'  저신뢰: {len(low_conf):,} ({len(low_conf)/len(df)*100:.1f}%)')

    # 고신뢰 정확도
    high_correct = (high_conf['bert_top1_label'] == high_conf['intent_label']).sum()
    if len(high_conf) > 0:
        print(f'  고신뢰 정확도: {high_correct}/{len(high_conf)} = {high_correct/len(high_conf)*100:.1f}%')

    # 저신뢰 BERT 정확도
    low_correct = (low_conf['bert_top1_label'] == low_conf['intent_label']).sum()
    if len(low_conf) > 0:
        print(f'  저신뢰 BERT 정확도: {low_correct}/{len(low_conf)} = {low_correct/len(low_conf)*100:.1f}%')

    return high_conf, low_conf


high_conf_df, low_conf_df = split_by_confidence(test_with_bert)

=== Confidence 분할 (threshold=0.7) ===
  고신뢰: 168 (4.5%)
  저신뢰: 3,569 (95.5%)
  고신뢰 정확도: 160/168 = 95.2%
  저신뢰 BERT 정확도: 1646/3569 = 46.1%


## 5. LLM 분류 (저신뢰 쿼리)
저신뢰 쿼리에 대해 Gemini가 BERT의 Top-K 후보 + Discriminative Description을 보고 최종 분류한다.

**프로세스**:
1. BERT Top-1과 LLM 예측이 일치 → 해당 의도로 확정
2. 불일치 → Comparative Reasoning으로 LLM이 최종 결정
3. 모든 후보 부적절 → null intent 판정

**체크포인트**: `checkpoints/loop/llm_predictions.parquet`

In [9]:
# === LLM 분류 프롬프트 ===

CLASSIFY_SYSTEM = """당신은 고객 상담 의도 분류 전문가입니다.

## 작업
주어진 쿼리(고객 의도 요약)를 후보 의도 중 하나로 분류하거나, 어느 것에도 해당하지 않으면 "null"로 판정하세요.

## 규칙
1. 각 후보 의도의 definition과 구별 기준(differs_from)을 참고하여 가장 적합한 의도를 선택하세요.
2. 소형 모델의 Top-1 예측이 함께 제공됩니다. 이를 참고하되, 독립적으로 판단하세요.
3. 어떤 후보 의도에도 해당하지 않으면 "null"을 선택하세요.
4. 선택한 의도와 근거를 JSON으로 출력하세요.

## 출력 형식 (JSON)
{
  "selected_intent": "행위-목적" 또는 "null",
  "reasoning": "선택 근거 한 문장",
  "agrees_with_bert": true/false
}"""


def build_classify_prompt(
    query: str,
    bert_top1: str,
    top_k_candidates: list,
    desc_map: dict,
) -> str:
    """
    LLM 분류 프롬프트를 구성한다.
    """
    parts = [f'쿼리: {query}', f'소형 모델 Top-1 예측: {bert_top1}', '', '후보 의도:']

    for cand in top_k_candidates:
        label = cand['label']
        prob = cand['prob']
        desc = desc_map.get(label, {})
        definition = desc.get('definition', '')

        parts.append(f'\n  [{label}] (신뢰도: {prob})')
        if definition:
            parts.append(f'    정의: {definition}')

        differs = desc.get('differs_from', [])
        if isinstance(differs, str):
            try:
                differs = json.loads(differs)
            except json.JSONDecodeError:
                differs = []
        # Top-K 내 의도와의 구별 기준만 포함
        cand_labels = {c['label'] for c in top_k_candidates}
        relevant_diffs = [d for d in differs if d.get('intent') in cand_labels]
        for d in relevant_diffs[:2]:  # 최대 2개
            parts.append(f'    ↔ {d["intent"]}와 구별: {d["distinction"]}')

    if CONFIG['null_intent_enabled']:
        parts.append('\n  [null] (위 의도 중 어느 것에도 해당하지 않음)')

    return '\n'.join(parts)


print('LLM 분류 프롬프트 정의 완료')

LLM 분류 프롬프트 정의 완료


In [10]:
# === Description 매핑 구축 ===

def build_desc_map(desc_df: pd.DataFrame) -> dict:
    """
    intent_label → {definition, differs_from} 매핑을 구축한다.
    """
    desc_map = {}
    for _, row in desc_df.iterrows():
        label = nfc(row['intent'])
        differs = row['differs_from']
        if isinstance(differs, str):
            try:
                differs = json.loads(differs)
            except json.JSONDecodeError:
                differs = []
        desc_map[label] = {
            'definition': row['definition'],
            'differs_from': differs,
        }
    print(f'Description 매핑: {len(desc_map)}개')
    return desc_map


desc_map = build_desc_map(desc_df)

Description 매핑: 906개


In [11]:
# === LLM 분류 실행 ===

def run_llm_classification(low_conf_df: pd.DataFrame, desc_map: dict) -> pd.DataFrame:
    """
    저신뢰 쿼리에 대해 LLM 분류를 실행한다.
    체크포인트가 존재하면 로드.
    """
    ckpt_path = PATH_CONFIG['loop_ckpt_dir'] / 'llm_predictions.parquet'
    if ckpt_path.exists():
        print(f'LLM 예측 체크포인트 로드: {ckpt_path}')
        return pd.read_parquet(ckpt_path)

    results = []
    for idx, row in tqdm(low_conf_df.iterrows(), total=len(low_conf_df), desc='LLM 분류'):
        query = row['input_text']
        bert_top1 = row['bert_top1_label']
        top_k = json.loads(row['bert_top_k'])

        prompt = build_classify_prompt(query, bert_top1, top_k, desc_map)
        response = call_gemini(prompt, CLASSIFY_SYSTEM)

        llm_label = 'null'
        reasoning = ''
        agrees = False

        if response:
            try:
                parsed = json.loads(response)
                llm_label = nfc(parsed.get('selected_intent', 'null'))
                reasoning = parsed.get('reasoning', '')
                agrees = parsed.get('agrees_with_bert', False)
            except json.JSONDecodeError:
                llm_label = 'PARSE_FAIL'

        results.append({
            'llm_label': llm_label,
            'llm_reasoning': reasoning,
            'llm_agrees_bert': agrees,
        })

    result_df = low_conf_df.reset_index(drop=True).copy()
    llm_df = pd.DataFrame(results)
    result_df = pd.concat([result_df, llm_df], axis=1)

    result_df.to_parquet(ckpt_path, index=False)
    print(f'LLM 예측 저장: {ckpt_path}')

    return result_df


low_conf_with_llm = run_llm_classification(low_conf_df, desc_map)

LLM 분류:   0%|          | 0/3569 [00:00<?, ?it/s]

LLM 예측 저장: /content/drive/MyDrive/5stone_experiment/1_clustering_test/checkpoints/loop/llm_predictions.parquet


## 6. 최종 분류 결정 통합
고신뢰(BERT) + 저신뢰(LLM) 결과를 통합하고 최종 분류를 결정한다.

**결정 로직**:
- 고신뢰: BERT Top-1 그대로 확정
- 저신뢰 + LLM이 BERT와 일치: 해당 의도로 확정
- 저신뢰 + LLM이 BERT와 불일치: LLM 결과 채택 (Comparative Reasoning)
- 저신뢰 + LLM이 null 판정: null intent로 분류

In [12]:
# === 최종 통합 ===

def merge_final_predictions(
    high_conf_df: pd.DataFrame,
    low_conf_with_llm: pd.DataFrame,
) -> pd.DataFrame:
    """
    고신뢰 + 저신뢰 결과를 통합하여 최종 분류 결과를 생성한다.
    """
    # 고신뢰: BERT 결과 확정
    high = high_conf_df.copy()
    high['final_label'] = high['bert_top1_label']
    high['decision_source'] = 'bert_high_conf'
    high['is_null'] = False

    # 저신뢰: LLM 기반 결정
    low = low_conf_with_llm.copy()
    final_labels = []
    decision_sources = []
    is_nulls = []

    for _, row in low.iterrows():
        llm_label = row.get('llm_label', 'null')
        bert_label = row['bert_top1_label']
        agrees = row.get('llm_agrees_bert', False)

        if llm_label == 'null':
            final_labels.append('NULL_INTENT')
            decision_sources.append('llm_null')
            is_nulls.append(True)
        elif llm_label == 'PARSE_FAIL':
            final_labels.append(bert_label)  # fallback to BERT
            decision_sources.append('bert_fallback')
            is_nulls.append(False)
        elif agrees or llm_label == bert_label:
            final_labels.append(llm_label)
            decision_sources.append('bert_llm_agree')
            is_nulls.append(False)
        else:
            # 불일치 → LLM 우선 (Comparative Reasoning)
            final_labels.append(llm_label)
            decision_sources.append('llm_override')
            is_nulls.append(False)

    low['final_label'] = final_labels
    low['decision_source'] = decision_sources
    low['is_null'] = is_nulls

    # 통합
    # 누락 컬럼 맞추기
    for col in ['llm_label', 'llm_reasoning', 'llm_agrees_bert']:
        if col not in high.columns:
            high[col] = ''

    common_cols = [
        'source_id', 'source', 'consulting_category', 'split',
        'intent_summary', 'intent_label', 'cluster_id', 'input_text',
        'bert_top1_label', 'bert_confidence', 'bert_top_k',
        'llm_label', 'llm_reasoning', 'llm_agrees_bert',
        'final_label', 'decision_source', 'is_null',
    ]
    # 존재하는 컬럼만 선택
    high_cols = [c for c in common_cols if c in high.columns]
    low_cols = [c for c in common_cols if c in low.columns]

    combined = pd.concat([high[high_cols], low[low_cols]], ignore_index=True)

    # 통계
    print(f'=== 최종 분류 결과 ===')
    print(f'  총 Test 건수: {len(combined):,}')
    print(f'\n--- 결정 출처 분포 ---')
    for src, cnt in combined['decision_source'].value_counts().items():
        print(f'  {src}: {cnt:,} ({cnt/len(combined)*100:.1f}%)')
    print(f'\n  Null intent: {combined["is_null"].sum():,}')

    return combined


final_results = merge_final_predictions(high_conf_df, low_conf_with_llm)

=== 최종 분류 결과 ===
  총 Test 건수: 3,737

--- 결정 출처 분포 ---
  bert_llm_agree: 1,896 (50.7%)
  llm_override: 1,119 (29.9%)
  llm_null: 554 (14.8%)
  bert_high_conf: 168 (4.5%)

  Null intent: 554


## 7. 전체 성능 평가
고신뢰(BERT) + 저신뢰(LLM) 통합 분류의 정확도를 ground truth(intent_label)와 비교하여 측정한다.

In [13]:
# === 성능 평가 ===

def evaluate_loop(results_df: pd.DataFrame) -> dict:
    """
    Loop 전체 성능을 평가한다.
    """
    metrics = {}

    # null이 아닌 건만 정확도 계산 (null은 "기존 의도에 없다"는 판정이므로 별도)
    non_null = results_df[~results_df['is_null']].copy()
    null_df = results_df[results_df['is_null']].copy()

    # 전체 정확도 (non-null)
    correct = (non_null['final_label'] == non_null['intent_label']).sum()
    total = len(non_null)
    overall_acc = correct / total if total > 0 else 0

    print(f'=== Loop 성능 평가 ===')
    print(f'\n--- 전체 (non-null) ---')
    print(f'  정확도: {correct:,}/{total:,} = {overall_acc*100:.2f}%')
    metrics['overall_accuracy'] = round(overall_acc, 4)

    # BERT only baseline
    bert_correct = (results_df['bert_top1_label'] == results_df['intent_label']).sum()
    bert_acc = bert_correct / len(results_df)
    print(f'\n--- BERT only baseline ---')
    print(f'  정확도: {bert_correct:,}/{len(results_df):,} = {bert_acc*100:.2f}%')
    metrics['bert_only_accuracy'] = round(bert_acc, 4)

    # 결정 출처별 정확도
    print(f'\n--- 결정 출처별 정확도 ---')
    for src in sorted(non_null['decision_source'].unique()):
        sub = non_null[non_null['decision_source'] == src]
        src_correct = (sub['final_label'] == sub['intent_label']).sum()
        src_acc = src_correct / len(sub) if len(sub) > 0 else 0
        print(f'  {src}: {src_correct}/{len(sub)} = {src_acc*100:.1f}%')
        metrics[f'acc_{src}'] = round(src_acc, 4)

    # LLM 개선 효과
    improvement = overall_acc - bert_acc
    print(f'\n--- LLM 개선 효과 ---')
    print(f'  BERT only: {bert_acc*100:.2f}%')
    print(f'  BERT + LLM: {overall_acc*100:.2f}%')
    print(f'  개선: {improvement*100:+.2f}%p')
    metrics['improvement'] = round(improvement, 4)

    # Null intent 분석
    if len(null_df) > 0:
        print(f'\n--- Null Intent ---')
        print(f'  판정 건수: {len(null_df)}')
        # null로 판정된 건의 원래 라벨 분포
        print(f'  원래 라벨 분포 (상위 10):')
        for lbl, cnt in null_df['intent_label'].value_counts().head(10).items():
            print(f'    {lbl}: {cnt}')
        metrics['null_count'] = len(null_df)

    # LLM override 분석 (BERT와 불일치하여 LLM이 결정한 케이스)
    override = non_null[non_null['decision_source'] == 'llm_override']
    if len(override) > 0:
        override_correct = (override['final_label'] == override['intent_label']).sum()
        override_bert_would = (override['bert_top1_label'] == override['intent_label']).sum()
        print(f'\n--- LLM Override 분석 ---')
        print(f'  건수: {len(override)}')
        print(f'  LLM 정답: {override_correct} ({override_correct/len(override)*100:.1f}%)')
        print(f'  BERT였으면 정답: {override_bert_would} ({override_bert_would/len(override)*100:.1f}%)')
        net_gain = override_correct - override_bert_would
        print(f'  LLM 순이득: {net_gain:+d}건')
        metrics['override_llm_correct'] = int(override_correct)
        metrics['override_bert_would'] = int(override_bert_would)

    return metrics


metrics = evaluate_loop(final_results)

=== Loop 성능 평가 ===

--- 전체 (non-null) ---
  정확도: 1,854/3,183 = 58.25%

--- BERT only baseline ---
  정확도: 1,806/3,737 = 48.33%

--- 결정 출처별 정확도 ---
  bert_high_conf: 160/168 = 95.2%
  bert_llm_agree: 1239/1896 = 65.3%
  llm_override: 455/1119 = 40.7%

--- LLM 개선 효과 ---
  BERT only: 48.33%
  BERT + LLM: 58.25%
  개선: +9.92%p

--- Null Intent ---
  판정 건수: 554
  원래 라벨 분포 (상위 10):
    확인요청-결제방식: 7
    확인요청-항공권및여행상품정보: 7
    문의-풀빌라예약정보: 6
    추천요청-숙소: 6
    문의-도시가스자동이체변경: 5
    문의-할부개월수: 5
    해지요청-카드: 5
    문의-카드정보: 5
    문의-발리여행정보: 5
    문의-잔액부족: 5

--- LLM Override 분석 ---
  건수: 1119
  LLM 정답: 455 (40.7%)
  BERT였으면 정답: 238 (21.3%)
  LLM 순이득: +217건


## 8. 결과 저장

In [15]:
# === 결과 저장 ===

def save_loop_results(results_df: pd.DataFrame, metrics: dict) -> None:
    # 분류 결과
    output_path = PATH_CONFIG['loop_results']
    results_df.to_parquet(output_path, index=False)
    print(f'분류 결과 저장: {output_path}')

    # 평가 메트릭
    eval_path = PATH_CONFIG['loop_evaluation']
    metrics['config'] = {
        'bert_model': CONFIG['bert_model_name'],
        'confidence_threshold': CONFIG['confidence_threshold'],
        'top_k': CONFIG['top_k_candidates'],
        'null_intent_enabled': CONFIG['null_intent_enabled'],
    }
    with open(eval_path, 'w', encoding='utf-8') as f:
        json.dump(metrics, f, ensure_ascii=False, indent=2, default=str)
    print(f'평가 메트릭 저장: {eval_path}')


save_loop_results(final_results, metrics)

ArrowTypeError: ("Expected bytes, got a 'bool' object", 'Conversion failed for column llm_agrees_bert with type object')

## 9. 비용 정산

In [ ]:
# === 비용 정산 ===

def estimate_loop_cost(low_conf_count: int) -> None:
    est_input = low_conf_count * 900   # ~900 토큰/호출
    est_output = low_conf_count * 50
    cost_usd = (est_input / 1e6) * 0.10 + (est_output / 1e6) * 0.40
    cost_krw = cost_usd * 1450

    print(f'=== Loop 비용 추정 ===')
    print(f'  LLM 호출: {low_conf_count:,}건')
    print(f'  입력 토큰: ~{est_input:,.0f}')
    print(f'  출력 토큰: ~{est_output:,.0f}')
    print(f'  비용: ${cost_usd:.3f} (₩{cost_krw:,.0f})')

    total_spent = 5200 + 1000 + 160 + cost_krw
    print(f'\n  누적: ₩{total_spent:,.0f} / ₩50,000')
    print(f'  잔여: ₩{50000 - total_spent:,.0f}')


estimate_loop_cost(len(low_conf_df))

## 10. 세션 종료
세션을 자동으로 종료하고, 자원을 반환한다.

In [ ]:
from google.colab import runtime
runtime.unassign()